In [ ]:
# bowaka_v2_lab notebook bootstrap cell — DO NOT EDIT BY HAND.
# Adds the lab's src/ to sys.path so `import bowaka_v2_lab` works regardless of
# how the notebook is launched (jupyter / papermill / pytest).
import os
import sys
from pathlib import Path

_here = Path.cwd()
for _candidate in [_here, *_here.parents]:
    if (_candidate / "src" / "bowaka_v2_lab" / "__init__.py").is_file():
        sys.path.insert(0, str(_candidate / "src"))
        break
import bowaka_v2_lab  # noqa: F401
print("bowaka_v2_lab", bowaka_v2_lab.__version__)


In [ ]:
# Papermill parameter cell.
CONFIG_PATH = 'research_notebooks/bowaka_v2_lab/configs/bowaka_v2_backtest_smoke.yml'


# 04 — Intraday Event Replay

In [ ]:
import datetime as _dt
import pandas as pd
from pathlib import Path
from bowaka_v2_lab.config import load_config
from bowaka_v2_lab.scanner.replay import replay_scanner
from bowaka_v2_lab.sim.replay_fixtures import synthetic_universe, synthetic_daily_cache
cfg = load_config(CONFIG_PATH)
syms = cfg.get('universe', {}).get('symbols') or ['AAA','BBB','CCC']
universe = synthetic_universe(syms)
daily_cache = synthetic_daily_cache(syms)
scan_ts = [pd.Timestamp('2024-09-04 14:00:00', tz='UTC')]
def supplier(sym, ts):
    # Minimal fixture bars.
    rows = []
    for i in range(30):
        rows.append({'timestamp': pd.Timestamp('2024-09-04 13:30:00', tz='UTC') + pd.Timedelta(minutes=i),
                      'open': 100+i*0.1, 'high': 100+i*0.2, 'low': 99.5, 'close': 100+i*0.15, 'volume': 1000.0})
    return pd.DataFrame(rows)
run_dir = Path('artifacts/runs/replay_nb_smoke')
summary = replay_scanner(cfg=cfg, universe_snapshot=universe, daily_cache=daily_cache,
  volume_curve=None, scan_timestamps=scan_ts, bars_supplier=supplier, run_dir=run_dir)
print(summary)
